# Синтетический AA-тест методом Монте-Карло

### Импортируем нужные пакеты

In [1]:
import pandahouse as ph
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind
from tqdm import tqdm # для отслеживания прогресса при симуляции данных и рассчёте t-test


In [2]:
rng = np.random.default_rng() # для более оптимизированной генерации выборок (меньше памяти)

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

connection = {
    'host': os.getenv('CLICKHOUSE_HOST'),
    'database': os.getenv('CLICKHOUSE_DATABASE'),
    'user': os.getenv('CLICKHOUSE_USER'),
    'password': os.getenv('CLICKHOUSE_PASSWORD')
}

## Проведём синтетический АА-тест, срок - 1 неделя

### Выгружаем нужные данные по просмотрам

In [ ]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])
views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum()

In [5]:
views.head()

,views,users,p
0,198,25,0.000595
1,116,159,0.003786
2,66,336,0.008001
3,272,2,0.000048
4,240,7,0.000167


In [7]:
print(f"{views.users.sum():.0f}")

41997


In [9]:
print(f"{views.users.sum()//2:.0f}") # размер выборки для каждой группы, так как делим ровно 50/50

20998


### Выгружаем данные по CTR

In [10]:
q = """
SELECT 
   floor(ctr, 2) as ctr, 
   count() as users
FROM (SELECT toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
FROM {db}.feed_actions
WHERE dt between '2025-11-14' and '2025-11-20'
GROUP BY dt, user_id
)
GROUP BY ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum() # Основа для выборок

In [11]:
ctr.head()

,ctr,users,p
0,0.00,1443,0.016952
1,0.65,4,0.000047
2,0.71,5,0.000059
3,0.49,4,0.000047
4,0.54,72,0.000846


### Считаем t-test и мощность

In [14]:
#ttest

beta = 0

sample_size = views.users.sum() // 2 # 20998


for i in tqdm(range(20000)):
    
    # пусть group_A будет тестовой группой. Тогда создадим выборку по просмотрам для тестовой группы:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # Изменим данные с поправкой на эффект алгоритма (по условию от команды ML) для тестовой группы:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 50)
    
    
    # group_B - контрольная. Создадим выборку для неё:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # создадим выборку с CTR для каждой группы:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # теперь симулируем распределение лайков для каждой группы:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # считаем t-test с поправкой Уэлча, "складываем" мощности значимых t-test (p-value <= 0.05) в объект beta:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# считаем итоговую мощность по полученным тестам:
result = (beta / 20000) * 100
print(f"{result:.4f}")

100%|██████████| 20000/20000 [02:32<00:00, 131.04it/s]

25.7550


Мощность теста получается меньше 30%. То есть, при заданных условиях мы сможем "поймать" эффект (если он действительно есть) от нового алгоритма только в 30% случаев.

### А если алгоритм срабатывает на пользователях с просмотрами 30 и выше?

Позже, ML-команда доработала алгоритм, и теперь он работает для пользователей от 30 просмотров ленты. Поэтому рассчитаем мощность для потенциального эффекта с поправкой на это улучшение:

In [15]:
#ttest

beta = 0

for i in tqdm(range(20000)):
    
    # пусть group_A будет тестовой группой. Тогда создаздим выборку по просмотрам для тестовой группы:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # Изменим данные с поправкой на эффект алгоритма (по условию от команды ML) для тестовой группы:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B - контрольная. Создадим выборку для неё:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # создадим выборку с CTR для каждой группы:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # теперь симулируем распределение лайков для каждой группы:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # считаем t-test с поправкой Уэлча, "складываем" мощности значимых t-test (p-value <= 0.05) в объект beta:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# считаем итоговую мощность по полученным тестам:
result = (beta / 20000) * 100
print(f"{result:.4f}")

100%|██████████| 20000/20000 [02:33<00:00, 129.88it/s]

41.4900


Видим, что, действительно, мощность теста повысилась. Теперь мы можем обнаружить эффект уже в ~41% случаев.

## Теперь что, если наш тест длится 2 недели?

По условию от заказчика, надо также рассчитать мощность теста для более долгого срока: допустим, A/B-тест будет идти 2 недели.

Также есть допущение, что в эти две недели придёт столько же пользователей, сколько пришло суммарно за период АА-теста и АБ-теста. Учтём это при рассчёте выборки для  AA-теста.

### Выгружаем нужные данные по просмотрам

In [16]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum()

In [17]:
views.sort_values(by = 'p', ascending = True)

,views,users,p
238,316,1,0.000024
280,301,1,0.000024
139,287,1,0.000024
137,255,1,0.000024
30,271,1,0.000024
...,...,...,...
16,30,469,0.011167
265,35,485,0.011548
116,14,500,0.011906
49,15,537,0.012787


### Выгружаем данные по CTR

In [18]:
q = """
select 
   floor(ctr, 2) as ctr, 
   count() as users
from (select toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from {db}.feed_actions
where dt between '2025-11-14' and '2025-11-20'
group by dt, user_id
)
group by ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum()

In [19]:
ctr.sort_values(by = 'ctr', ascending = True)

,ctr,users,p
0,0.00,1443,0.016952
69,0.02,48,0.000564
23,0.03,142,0.001668
32,0.04,312,0.003665
16,0.05,727,0.008541
...,...,...,...
12,0.81,2,0.000023
73,0.83,1,0.000012
26,0.85,3,0.000035
39,0.88,1,0.000012


### Выгружаем данные за 2 недели, чтобы посчитать размер выборки по этому условию

In [20]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-27'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views_2 = ph.read_clickhouse(q, connection = connection)

### Считаем t-test и мощность

In [21]:
#ttest

sample_size = views_2.users.sum() // 2 # 30591

beta = 0

for i in tqdm(range(20000)):
    
    # пусть group_A будет тестовой группой. Тогда создадим выборку по просмотрам для тестовой группы:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # Изменим данные с поправкой на эффект алгоритма (по условию от команды ML) для тестовой группы:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B - контрольная. Создадим выборку для неё:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # создадим выборку с CTR для каждой группы:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # теперь симулируем распределение лайков для каждой группы:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # считаем t-test с поправкой Уэлча, "складываем" мощности значимых t-test (p-value <= 0.05) в объект beta:
    beta += (stats.ttest_ind(group_B_likes, group_A_likes, equal_var = False).pvalue <= 0.05)

# считаем итоговую мощность по полученным тестам:
result = (beta / 20000) * 100
print(f"{result:.4f}")

100%|██████████| 20000/20000 [03:49<00:00, 87.14it/s]

56.2850


С увеличением срока проведения теста (и, следовательно, увеличением выборки) мощность ещё больше выросла. Теперь значимый эффект (если есть) будет обнаружен в ~56% случаев.

## Рассчёт мощности только для тех пользователей, на которых алгоритм окажет эффект

Всё это время мы анализировали наши выборки целиком — и тех пользователей, на которых алгоритм повлиял, и тех, кого он не мог затронуть (меньше 30 просмотров). Заказчик попросил исправить рассчёты мощности, посчитав t-test только для нужных пользователей. Логика такая: да, выборка будет меньше, но мы избавимся от мусора — а значит, и чувствительность наверняка будет выше.

Остальные условия остаются неизменными: смотрим с рассчётом срока теста на 2 недели и эффектом алгоритма для пользователей с 30+ просмотров.

### Выгрузим данные по просмотрам

In [22]:
q = '''
SELECT 
    views, 
    count() as users
FROM
(
SELECT 
    user_id, 
    sum(action = 'view') as views
FROM {db}.feed_actions
WHERE time::date BETWEEN '2025-11-14' AND '2025-11-20'
GROUP BY user_id
)
GROUP BY views
'''.format(db=connection['database'])

views = ph.read_clickhouse(q, connection = connection)
views['p'] = views.users / views.users.sum()

### Выгрузим данные по CTR

In [23]:
q = """
select 
   floor(ctr, 2) as ctr, 
   count() as users
from (select toDate(time) as dt, 
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from {db}.feed_actions
where dt between '2025-11-14' and '2025-11-20'
group by dt, user_id
)
group by ctr
""".format(db=connection['database'])

ctr = ph.read_clickhouse(q, connection = connection)
ctr['p'] = ctr.users / ctr.users.sum()

### Считаем t-test и мощность

In [24]:
#ttest

sample_size = views_2.users.sum() // 2 # 30591

beta = 0

for i in tqdm(range(20000)):
    
    # пусть group_A будет тестовой группой. Тогда создадим выборку по просмотрам для тестовой группы:
    group_A_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # Изменим данные с поправкой на эффект алгоритма (по условию от команды ML) для тестовой группы:
    group_A_views = group_A_views + ((1 + rng.binomial(1, 0.5, size = sample_size)) * rng.binomial(1, 0.9, size = sample_size)) * (group_A_views >= 30)
    
    
    # group_B - контрольная. Создадим выборку для неё:
    group_B_views = rng.choice(views['views'], 
                           size = sample_size, 
                           replace = True, 
                           p = views['p']).astype(np.int64)
    
    # создадим выборку с CTR для каждой группы:
    group_A_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    group_B_ctr = rng.choice(ctr['ctr'], size = sample_size, replace = True, p = ctr['p'])
    
    # теперь симулируем распределение лайков для каждой группы:
    group_A_likes = rng.binomial(group_A_views, group_A_ctr)
    group_B_likes = rng.binomial(group_B_views, group_B_ctr)
    
    # создаём фильтры, чтобы провести итоговый тест на пользователях с 30+ просмотров
    mask_A = group_A_views >= 30
    mask_B = group_B_views >= 30
    
    # считаем t-test с поправкой Уэлча, "складываем" результаты значимых t-test (p-value <= 0.05) в объект beta:
    beta += (stats.ttest_ind(group_B_likes[mask_B], group_A_likes[mask_A], equal_var = False).pvalue <= 0.05)

# считаем итоговую мощность по полученным тестам:
result = (beta / 20000) * 100
print(f"{result:.4f}")


100%|██████████| 20000/20000 [03:49<00:00, 87.16it/s]

64.5800


В итоге получаем мощность теста 64%. Таким образом, даже если смотреть эффект алгоритма на пользователях с 30+ просмотров, мощность хоть и повышается, но остаётся недостаточной: только в 64% случаев мы будем обнаруживать эффект от нового алгоритма (что ниже стандартных 80%).

# Рекомендации

По результатам анализа основная рекомендация - **не запускать** AB-тест с текущими параметрами алгоритма и дизайном эксперимента. Возможные следующие шаги:
* увеличить длительность эксперимента или расширить долю трафика для накопления большей выборки
* продлить эксперимент на N дней для накопления достаточной статистической мощности
* дальнейшее сужение критериев отбора может повысить мощность за счёт гомогенизации выборки
* проработать гипотезу: возможно, текущая версия алгоритма даёт слишком малый прирост метрики
